# Trendline Breakout Binary Options Optimizer (Colab)

- Upload 1-minute OHLC data (CSV) with columns: `time,open,high,low,close`.
- The script reproduces breakout logic similar to the LuxAlgo trendline method.
- It will backtest RISE/FALL binary options that settle after N minutes and search for the best expiry.

Note: This notebook is for research/education. It does not constitute financial advice.

In [ ]:
#@title Upload 1-minute CSV (time,open,high,low,close)
from google.colab import files
import io
import pandas as pd
uploaded = files.upload()
assert uploaded, "No file uploaded."
fname = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[fname]))
# Ensure required columns
required = ['time','open','high','low','close']
missing = [c for c in required if c not in df.columns]
assert not missing, f'Missing required columns: {missing}'
# Parse time if needed
if not pd.api.types.is_datetime64_any_dtype(df['time']):
    df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time').reset_index(drop=True)
df.head()

In [ ]:
#@title Parameters
length = 14 #@param {type:"integer"}
mult = 1.0 #@param {type:"number"}
calc_method = 'Atr' #@param ['Atr','Stdev','Linreg']
stake = 100.0 #@param {type:"number"}
payout_pct = 80.0 #@param {type:"number"}
min_expiry = 1 #@param {type:"integer"}
max_expiry = 60 #@param {type:"integer"}

assert min_expiry >= 1 and max_expiry >= min_expiry
print(f'Optimizing expiry minutes in range: {min_expiry}..{max_expiry}')

In [ ]:
#@title Breakout computation and optimizer
import numpy as np

def compute_slope(series, method, length, mult):
    if method == 'Atr':
        # Simple ATR approximation on close for speed: use high-low true range
        high = df['high'].to_numpy()
        low = df['low'].to_numpy()
        close = df['close'].to_numpy()
        prev_close = np.roll(close, 1)
        tr = np.maximum(high - low, np.maximum(np.abs(high - prev_close), np.abs(low - prev_close)))
        atr = pd.Series(tr).rolling(length, min_periods=length).mean().to_numpy()
        return (atr / length) * mult
    elif method == 'Stdev':
        return pd.Series(series).rolling(length, min_periods=length).std().to_numpy() / length * mult
    else:
        # Linreg approx: using variance of index to mimic pine formula
        n = np.arange(len(series))
        sma_x = pd.Series(n).rolling(length, min_periods=length).mean().to_numpy()
        sma_y = pd.Series(series).rolling(length, min_periods=length).mean().to_numpy()
        sma_xy = pd.Series(series*n).rolling(length, min_periods=length).mean().to_numpy()
        var_x = pd.Series(n).rolling(length, min_periods=length).var().to_numpy()
        with np.errstate(divide='ignore', invalid='ignore'):
            lin = np.abs(sma_xy - sma_y * sma_x) / np.where(var_x==0, np.nan, var_x) / 2.0 * mult
        return lin

def pivothigh(arr, left, right):
    # True when value equals rolling max with exact center
    s = pd.Series(arr)
    roll = s.rolling(window=left+right+1, center=True).max()
    return (s == roll).to_numpy()

def pivotlow(arr, left, right):
    s = pd.Series(arr)
    roll = s.rolling(window=left+right+1, center=True).min()
    return (s == roll).to_numpy()

def compute_breakouts(df, length, mult, method):
    close = df['close'].to_numpy()
    n = np.arange(len(close))
    slope = compute_slope(close, method, length, mult)
    ph = pivothigh(close, length, length)
    pl = pivotlow(close, length, length)
    upper = np.zeros_like(close, dtype=float)
    lower = np.zeros_like(close, dtype=float)
    slope_ph = np.zeros_like(close, dtype=float)
    slope_pl = np.zeros_like(close, dtype=float)

    for i in range(len(close)):
        if ph[i]:
            slope_ph[i] = slope[i] if not np.isnan(slope[i]) else (slope_ph[i-1] if i>0 else 0.0)
            upper[i] = close[i]
        else:
            slope_ph[i] = slope_ph[i-1] if i>0 else 0.0
            upper[i] = (upper[i-1] - slope_ph[i]) if i>0 else 0.0
        if pl[i]:
            slope_pl[i] = slope[i] if not np.isnan(slope[i]) else (slope_pl[i-1] if i>0 else 0.0)
            lower[i] = close[i]
        else:
            slope_pl[i] = slope_pl[i-1] if i>0 else 0.0
            lower[i] = (lower[i-1] + slope_pl[i]) if i>0 else 0.0

    upos = np.zeros_like(close, dtype=int)
    dnos = np.zeros_like(close, dtype=int)
    for i in range(len(close)):
        if ph[i]:
            upos[i] = 0
        else:
            upos[i] = 1 if close[i] > (upper[i] - slope_ph[i] * length) else (upos[i-1] if i>0 else 0)
        if pl[i]:
            dnos[i] = 0
        else:
            dnos[i] = 1 if close[i] < (lower[i] + slope_pl[i] * length) else (dnos[i-1] if i>0 else 0)

    up_break = (np.diff(np.r_[0, upos]) > 0)
    down_break = (np.diff(np.r_[0, dnos]) > 0)
    return up_break, down_break

def backtest_binary(df, up_break, down_break, expiry_mins, stake, payout_pct):
    close = df['close'].to_numpy()
    times = df['time'].to_numpy()
    pnl = 0.0
    wins = 0
    losses = 0
    i = 0
    n = len(close)
    # non-overlapping assumption, take next signal only after settlement
    while i < n:
        if up_break[i] or down_break[i]:
            entry_price = close[i]
            entry_time = times[i]
            # find settlement index where time >= entry_time + expiry_mins
            target_time = entry_time + pd.Timedelta(minutes=int(expiry_mins))
            j = i
            while j < n and times[j] < target_time:
                j += 1
            if j >= n:
                break
            settle_price = close[j]
            is_rise = bool(up_break[i])
            is_win = (settle_price > entry_price) if is_rise else (settle_price < entry_price)
            if is_win:
                pnl += stake * (payout_pct/100.0)
                wins += 1
            else:
                pnl -= stake
                losses += 1
            i = j + 1
        else:
            i += 1
    return pnl, wins, losses

# Compute signals once
up_break, down_break = compute_breakouts(df, length, mult, calc_method)

# Grid search expiries
results = []
for exp in range(int(min_expiry), int(max_expiry)+1):
    pnl, wins, losses = backtest_binary(df, up_break, down_break, exp, stake, payout_pct)
    results.append((exp, pnl, wins, losses))

res_df = pd.DataFrame(results, columns=['expiry_minutes','pnl','wins','losses'])
res_df['trades'] = res_df['wins'] + res_df['losses']
res_df['winrate'] = np.where(res_df['trades']>0, res_df['wins']/res_df['trades'], np.nan)
res_df.sort_values('pnl', ascending=False).head(10)

In [ ]:
#@title Plot best expiry performance
import matplotlib.pyplot as plt
best = res_df.sort_values('pnl', ascending=False).iloc[0]
print('Best expiry (min):', int(best['expiry_minutes']), 'PNL:', best['pnl'], 'Trades:', int(best['trades']), 'Winrate:', round(best['winrate']*100,2) if pd.notna(best['winrate']) else None)
fig, ax = plt.subplots(1,2, figsize=(12,4))
ax[0].plot(res_df['expiry_minutes'], res_df['pnl'], '-o')
ax[0].set_title('PNL vs Expiry (min)')
ax[0].set_xlabel('Expiry (minutes)')
ax[0].set_ylabel('PNL')
ax[1].plot(res_df['expiry_minutes'], res_df['winrate']*100, '-o')
ax[1].set_title('Winrate vs Expiry (min)')
ax[1].set_xlabel('Expiry (minutes)')
ax[1].set_ylabel('Winrate %')
plt.show()